# 🤖 MediaPipe — Introducción al Framework

**MediaPipe** es un framework de código abierto desarrollado por **Google** especializado en la creación de *pipelines de Machine Learning multimodales en tiempo real*.

Su arquitectura está basada en **grafos acíclicos dirigidos (DAG)**, lo que permite un procesamiento de baja latencia ideal para aplicaciones interactivas con cámara.

---

## 📦 Módulos principales

| Módulo | Descripción | Puntos detectados |
|--------|-------------|-------------------|
| `hands` | Detección de manos | 21 landmarks 3D por mano |
| `pose` | Detección de cuerpo completo | 33 landmarks 3D |
| `face_detection` | Detección de rostros | Bounding box + keypoints |
| `face_mesh` | Malla facial detallada | 468 landmarks 3D |

> **Pipeline común:** Cada módulo combina un *modelo detector* (localiza la región de interés) y un *modelo de landmarks* (mapea la superficie en detalle), todo optimizado para GPU.

## ⚙️ Instalación e importación de librerías

In [1]:
# Instalar dependencias (ejecutar solo si no están instaladas)

!pip install mediapipe==0.10.14 opencv-python


ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.14 (from versions: 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for mediapipe==0.10.14


In [2]:
print("Hola MUndo")

Hola MUndo


In [3]:
import cv2
import sys
import mediapipe as mp

print("✅ Librerías importadas con éxito")
print(f"   MediaPipe versión : {mp.__version__}")
print(f"   OpenCV versión    : {cv2.__version__}")
print(f"   Python versión    : {sys.version}")

✅ Librerías importadas con éxito
   MediaPipe versión : 0.10.35
   OpenCV versión    : 4.13.0
   Python versión    : 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]


---

# ✋ Módulo 1 — Detección de Manos (`hands`)

## ¿Cómo funciona?

MediaPipe Hands detecta hasta **2 manos simultáneamente** y traza **21 landmarks en 3D** por mano.

El pipeline usa dos modelos encadenados:
1. **Detector de palma** — localiza la mano en la imagen.
2. **Modelo de landmarks** — mapea los 21 puntos a partir del recorte de la palma.

En el seguimiento continuo se prioriza reutilizar los landmarks del frame anterior; si el seguimiento falla, se reactiva el detector de palma.

### Mapa de landmarks

| ID | Parte | ID | Parte |
|----|-------|----|-------|
| 0 | Muñeca | 9–12 | Dedo medio |
| 1–4 | Pulgar | 13–16 | Anular |
| 5–8 | Índice | 17–20 | Meñique |

## Ejemplo 1A — Detección básica con todos los puntos 1 Mano

In [4]:
import cv2
import mediapipe as mp


# Inicializar MediaPipe Hands
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,  # Solo una mano
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Abrir cámara
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("No se pudo acceder a la cámara")
        break

    # Voltear imagen tipo espejo
    frame = cv2.flip(frame, 1)

    # Convertir BGR a RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Procesar imagen
    resultado = hands.process(rgb)

    # Dibujar puntos y conexiones
    if resultado.multi_hand_landmarks:
        for mano in resultado.multi_hand_landmarks:
            mp_draw.draw_landmarks(
                frame,
                mano,
                mp_hands.HAND_CONNECTIONS
            )

    cv2.imshow("Reconocimiento de Mano", frame)

    # Salir con ESC
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [10]:
py -0p

SyntaxError: invalid decimal literal (1988559037.py, line 1)

In [15]:
import cv2
import mediapipe as mp

# Inicializar MediaPipe
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,  # Detectar hasta 2 manos
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Abrir cámara
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("No se pudo acceder a la cámara")
        break

    # Efecto espejo
    frame = cv2.flip(frame, 1)

    # Convertir a RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Procesar imagen
    resultado = hands.process(rgb)

    # Dibujar manos y mostrar etiqueta
    if resultado.multi_hand_landmarks and resultado.multi_handedness:

        for mano, etiqueta in zip(
            resultado.multi_hand_landmarks,
            resultado.multi_handedness
        ):

            # Obtener tipo de mano
            tipo = etiqueta.classification[0].label

            # Coordenadas del punto 0 (muñeca)
            h, w, c = frame.shape
            x = int(mano.landmark[0].x * w)
            y = int(mano.landmark[0].y * h)

            # Dibujar puntos y conexiones
            mp_draw.draw_landmarks(
                frame,
                mano,
                mp_hands.HAND_CONNECTIONS
            )

            # Mostrar texto
            cv2.putText(
                frame,
                tipo,
                (x - 20, y - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2
            )

    cv2.imshow("Deteccion de Dos Manos", frame)

    # Salir con ESC
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [16]:
import cv2
import mediapipe as mp

# Inicializar MediaPipe Hands
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,  # Solo una mano
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Abrir cámara
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("No se pudo acceder a la cámara")
        break

    # Voltear imagen tipo espejo
    frame = cv2.flip(frame, 1)

    # Convertir BGR a RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Procesar imagen
    resultado = hands.process(rgb)

    # Dibujar puntos y conexiones
    if resultado.multi_hand_landmarks:
        for mano in resultado.multi_hand_landmarks:
            mp_draw.draw_landmarks(
                frame,
                mano,
                mp_hands.HAND_CONNECTIONS
            )

    cv2.imshow("Reconocimiento de Mano", frame)

    # Salir con ESC
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [17]:
import cv2
import mediapipe as mp

# ─────────────────────────────────────────────────────────────
# Inicialización de MediaPipe
# ─────────────────────────────────────────────────────────────
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ─────────────────────────────────────────────────────────────
# Captura de cámara
# ─────────────────────────────────────────────────────────────
RUN_WEBCAM = True

cap = cv2.VideoCapture(0) if RUN_WEBCAM else None

# Verificar si la cámara abrió correctamente
if RUN_WEBCAM and not cap.isOpened():
    print("Error: No se pudo acceder a la cámara")
    raise SystemExit()

# ─────────────────────────────────────────────────────────────
# Bucle principal
# ─────────────────────────────────────────────────────────────
while RUN_WEBCAM:

    ret, frame = cap.read()

    if not ret:
        print("Error al capturar frame")
        break

    # Efecto espejo
    frame = cv2.flip(frame, 1)

    # Obtener dimensiones
    h, w, _ = frame.shape

    # Convertir BGR → RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Optimización de rendimiento
    rgb.flags.writeable = False

    # Procesamiento de manos
    results = hands.process(rgb)

    rgb.flags.writeable = True

    # ─────────────────────────────────────────────────────────
    # Dibujar landmarks
    # ─────────────────────────────────────────────────────────
    if results.multi_hand_landmarks:

        for hand_landmarks in results.multi_hand_landmarks:

            # Dibujar puntos y conexiones
            mp_draw.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

            # Recorrer los 21 puntos
            for id, lm in enumerate(hand_landmarks.landmark):

                # Coordenadas en píxeles
                x = int(lm.x * w)
                y = int(lm.y * h)

                # Dibujar punto
                cv2.circle(frame, (x, y), 5,
                           (255, 0, 255),
                           cv2.FILLED)

                # Mostrar ID del punto
                cv2.putText(frame,
                            str(id),
                            (x, y),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.5,
                            (0, 255, 0),
                            1)

            # ─────────────────────────────────────────────
            # Ejemplo: detectar índice levantado
            # ─────────────────────────────────────────────
            if hand_landmarks.landmark[8].y < hand_landmarks.landmark[6].y:

                cv2.putText(
                    frame,
                    "Indice levantado",
                    (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    2
                )

    # Mostrar ventana
    cv2.imshow("Detector de manos", frame)

    # Salir con tecla Q
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# ─────────────────────────────────────────────────────────────
# Liberar recursos
# ─────────────────────────────────────────────────────────────
if RUN_WEBCAM:
    cap.release()
    cv2.destroyAllWindows()
hands.close()

#Abrir camara


## Ejemplo 1B — Mostrar solo puntos seleccionados

En lugar de dibujar los 21 puntos, se puede filtrar cuáles mostrar. Útil cuando solo se necesita rastrear partes específicas (p. ej., únicamente las yemas de los dedos).

In [18]:
import cv2
import mediapipe as mp

# ── Inicialización ────────────────────────────────────────────────────────────
mp_hands = mp.solutions.hands
hands    = mp_hands.Hands()
RUN_WEBCAM = True

cap      = cv2.VideoCapture(0) if RUN_WEBCAM else None

# ── Configuración de puntos visibles ─────────────────────────────────────────
# Solo se dibujarán los IDs incluidos en esta lista
puntos_visibles = [
    0,             # muñeca
    1, 2, 3, 4,    # pulgar (base → punta)
    5, 8,          # índice (nudillo → punta)
    9, 12,         # medio
    13, 16,        # anular
    17, 20         # meñique
]

while RUN_WEBCAM:
    ret, frame = cap.read()
    if not ret:
        break

    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    if results.multi_hand_landmarks:
        h, w, _ = frame.shape

        for hand_landmarks in results.multi_hand_landmarks:
            for id, lm in enumerate(hand_landmarks.landmark):
                if id in puntos_visibles:
                    x = int(lm.x * w)
                    y = int(lm.y * h)
                    cv2.circle(frame, (x, y), 6, (0, 255, 0), -1)  # Punto verde

    cv2.imshow("Manos — Puntos seleccionados", frame)

    if cv2.waitKey(1) == ord("q"):
        break

if RUN_WEBCAM:
    cap.release()
    cv2.destroyAllWindows()
hands.close()

## Ejemplo 1C — Puntos seleccionados con conexiones personalizadas

Se define un esqueleto propio estableciendo manualmente las conexiones entre puntos, en lugar de usar `HAND_CONNECTIONS`.

In [19]:
import cv2
import mediapipe as mp

# ── Inicialización ────────────────────────────────────────────────────────────
mp_hands = mp.solutions.hands
hands    = mp_hands.Hands()
RUN_WEBCAM = True

cap      = cv2.VideoCapture(0) if RUN_WEBCAM else None

# ── Puntos y conexiones personalizados ───────────────────────────────────────
puntos_visibles = [0, 1, 2, 3, 4, 5, 8, 9, 12, 13, 16, 17, 20]

conexiones = [
    (0, 1),  (1, 2),  (2, 3),  (3, 4),   # pulgar
    (0, 5),  (5, 8),                      # índice
    (0, 9),  (9, 12),                     # medio
    (0, 13), (13, 16),                    # anular
    (0, 17), (17, 20)                     # meñique
]

while RUN_WEBCAM:
    ret, frame = cap.read()
    if not ret:
        break

    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    if results.multi_hand_landmarks:
        h, w, _ = frame.shape

        for hand_landmarks in results.multi_hand_landmarks:
            coordenadas = {}

            # 1. Guardar coordenadas de los puntos seleccionados
            for id, lm in enumerate(hand_landmarks.landmark):
                if id in puntos_visibles:
                    x, y = int(lm.x * w), int(lm.y * h)
                    coordenadas[id] = (x, y)
                    cv2.circle(frame, (x, y), 6, (0, 255, 0), -1)  # Punto verde

            # 2. Dibujar líneas entre los puntos conectados
            for inicio, fin in conexiones:
                if inicio in coordenadas and fin in coordenadas:
                    cv2.line(
                        frame,
                        coordenadas[inicio],
                        coordenadas[fin],
                        (255, 255, 255),  # Color blanco
                        2                 # Grosor
                    )

    cv2.imshow("Manos — Esqueleto personalizado", frame)

    if cv2.waitKey(1) == ord("q"):
        break

if RUN_WEBCAM:
    cap.release()
    cv2.destroyAllWindows()
hands.close()

In [21]:
import cv2
import mediapipe as mp
import threading
import os

# --------------------------------------------------
# Función para hablar usando la voz de Windows
# --------------------------------------------------
def hablar(texto):
    comando = (
        'PowerShell -Command "Add-Type -AssemblyName System.Speech;'
        f'(New-Object System.Speech.Synthesis.SpeechSynthesizer).Speak(\'{texto}\')"'
    )
    os.system(comando)

# --------------------------------------------------
# Inicializar MediaPipe
# --------------------------------------------------
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

ultima_mano = ""

# --------------------------------------------------
# Cámara
# --------------------------------------------------
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        print("No se pudo abrir la cámara")
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    resultado = hands.process(rgb)

    if resultado.multi_hand_landmarks and resultado.multi_handedness:

        for mano, etiqueta in zip(
            resultado.multi_hand_landmarks,
            resultado.multi_handedness
        ):

            tipo = etiqueta.classification[0].label

            if tipo == "Left":
                texto = "Mano Izquierda"
            else:
                texto = "Mano Derecha"

            # Hablar solo cuando cambia la mano detectada
            if texto != ultima_mano:
                threading.Thread(
                    target=hablar,
                    args=(texto,),
                    daemon=True
                ).start()

                ultima_mano = texto

            # Coordenadas de la muñeca
            h, w, c = frame.shape
            x = int(mano.landmark[0].x * w)
            y = int(mano.landmark[0].y * h)

            # Dibujar landmarks
            mp_draw.draw_landmarks(
                frame,
                mano,
                mp_hands.HAND_CONNECTIONS
            )

            # Mostrar texto
            cv2.putText(
                frame,
                texto,
                (x - 20, y - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

    cv2.imshow("Reconocimiento de Manos", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()

---

# 🧘 Módulo 2 — Detección de Cuerpo Completo (`pose`)

## ¿Cómo funciona?

MediaPipe Pose detecta **33 landmarks en coordenadas 3D** que cubren el cuerpo humano completo:

| Zona | Landmarks |
|------|-----------|
| Cabeza y cara | 0–10 |
| Hombros y brazos | 11–22 |
| Torso | 23–24 |
| Piernas y pies | 25–32 |

> `Pose` es el punto de partida para **MediaPipe Holistic**, que estima primero la pose corporal y luego recorta las regiones de interés (ROI) para manos y cara, optimizando la resolución de cada módulo.

## Ejemplo 2 — Detección de pose corporal

In [31]:
import cv2
import mediapipe as mp

# ── Inicialización ────────────────────────────────────────────────────────────
mp_pose = mp.solutions.pose          # Módulo de detección corporal
pose    = mp_pose.Pose()
draw    = mp.solutions.drawing_utils  # Utilidad para dibujar

RUN_WEBCAM = True

cap = cv2.VideoCapture(0) if RUN_WEBCAM else None

while RUN_WEBCAM:
    ret, frame = cap.read()
    if not ret:
        break

    # Convertir a RGB para MediaPipe
    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # ── Procesamiento ─────────────────────────────────────────────────────────
    results = pose.process(rgb)

    # Dibujar esqueleto si se detecta cuerpo
    if results.pose_landmarks:
        draw.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS    # Conexiones predefinidas entre los 33 puntos
        )

    cv2.imshow("Pose — Cuerpo completo", frame)

    if cv2.waitKey(1) == ord("q"):
        break

if RUN_WEBCAM:
    cap.release()
    cv2.destroyAllWindows()
pose.close()

c:\Users\super\Desktop\MACHINE LEARNING\MACHINE LEARNING 2026\Vision_computadora\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [24]:
import cv2
import mediapipe as mp

# Inicialización
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = pose.process(rgb)

    if results.pose_landmarks:

        # Dibujar esqueleto
        draw.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            draw.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=3),
            draw.DrawingSpec(color=(255,0,0), thickness=2)
        )

        # Cantidad de puntos detectados
        cv2.putText(
            frame,
            "Puntos detectados: 33",
            (10, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,255),
            2
        )

        # Nariz (Landmark 0)
        nariz = results.pose_landmarks.landmark[0]

        h, w, _ = frame.shape

        x = int(nariz.x * w)
        y = int(nariz.y * h)

        cv2.circle(frame, (x, y), 10, (0,0,255), -1)

        cv2.putText(
            frame,
            "Nariz",
            (x+15, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,0,255),
            2
        )

    cv2.imshow("Deteccion Corporal", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
pose.close()

In [25]:
import cv2
import mediapipe as mp

# Inicialización
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = pose.process(rgb)

    mensaje = ""

    if results.pose_landmarks:

        draw.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

        landmarks = results.pose_landmarks.landmark

        # Hombro izquierdo y muñeca izquierda
        hombro_izq = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
        muneca_izq = landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]

        # Hombro derecho y muñeca derecha
        hombro_der = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
        muneca_der = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value]

        # Detectar brazo levantado
        if muneca_izq.y < hombro_izq.y:
            mensaje = "Brazo Izquierdo Arriba"

        if muneca_der.y < hombro_der.y:
            mensaje = "Brazo Derecho Arriba"

        if mensaje:

            cv2.putText(
                frame,
                mensaje,
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                3
            )

    cv2.imshow("Deteccion de Postura", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
pose.close()

---

# 😊 Módulo 3 — Detección Facial

MediaPipe ofrece dos herramientas complementarias para el rostro:

| Herramienta | Uso | Output |
|-------------|-----|--------|
| `face_detection` | Detectar si hay un rostro y su posición | ~6 keypoints + bounding box |
| `face_mesh` | Mapear la superficie facial en detalle | 468 landmarks 3D |

---

## Ejemplo 3A — Detección de rostros (`face_detection`)

In [28]:
import cv2
import mediapipe as mp

# ── Inicialización ────────────────────────────────────────────────────────────
mp_face = mp.solutions.face_detection  # Módulo de detección de rostros
face    = mp_face.FaceDetection(
    model_selection=0,            # 0 = rostros cercanos (<2m), 1 = rostros lejanos
    min_detection_confidence=0.5
)
draw = mp.solutions.drawing_utils

RUN_WEBCAM = True

cap = cv2.VideoCapture(0) if RUN_WEBCAM else None

while RUN_WEBCAM:
    ret, frame = cap.read()
    if not ret:
        break

    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face.process(rgb)

    # Dibujar bounding box y keypoints de cada rostro detectado
    if results.detections:
        for detection in results.detections:
            draw.draw_detection(frame, detection)

    cv2.imshow("Face Detection — Rostros", frame)

    if cv2.waitKey(1) == ord("q"):
        break

if RUN_WEBCAM:
    cap.release()
    cv2.destroyAllWindows()
face.close()

c:\Users\super\Desktop\MACHINE LEARNING\MACHINE LEARNING 2026\Vision_computadora\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


## Ejemplo 3B — Malla facial (`face_mesh`)

Detecta **468 puntos 3D** sobre la superficie del rostro. Con `refine_landmarks=True` se agregan puntos adicionales de precisión para iris y contornos de labios.

In [30]:
import cv2
import mediapipe as mp

# ── Inicialización ────────────────────────────────────────────────────────────
mp_face_mesh = mp.solutions.face_mesh
face_mesh    = mp_face_mesh.FaceMesh(
    static_image_mode=False,      # False = modo video (tracking continuo)
    max_num_faces=1,              # Número máximo de rostros a detectar
    refine_landmarks=True,        # Agrega puntos de iris y contornos de labios
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ── Colores de visualización (formato BGR) ────────────────────────────────────
COLOR_PUNTOS = (182, 89, 224)   # Morado claro
COLOR_IRIS   = (255, 100, 100)  # Rosa (para centros de los ojos)
CENTROS_OJOS = {33, 133, 362, 263}  # IDs de los centros de cada ojo

RUN_WEBCAM = True

cap = cv2.VideoCapture(0) if RUN_WEBCAM else None

while RUN_WEBCAM:
    ret, frame = cap.read()
    if not ret:
        break

    frame   = cv2.flip(frame, 1)  # Efecto espejo para uso natural
    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    if results.multi_face_landmarks:
        for landmarks in results.multi_face_landmarks:
            h, w, _ = frame.shape

            # Convertir landmarks normalizados a coordenadas en píxeles
            puntos = [
                (int(lm.x * w), int(lm.y * h))
                for lm in landmarks.landmark
            ]

            # Dibujar cada punto con color y tamaño según su tipo
            for i, (x, y) in enumerate(puntos):
                if i in CENTROS_OJOS:
                    radio, color = 3, COLOR_IRIS
                else:
                    radio, color = 2, COLOR_PUNTOS

                cv2.circle(frame, (x, y), radio,      color,          -1, cv2.LINE_AA)
                cv2.circle(frame, (x, y), radio // 2, (255, 255, 255), -1, cv2.LINE_AA)  # Brillo

    cv2.putText(frame, "Presiona Q para salir", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow("Face Mesh — Malla facial", frame)

    if cv2.waitKey(1) == ord("q"):
        break

if RUN_WEBCAM:
    cap.release()
    cv2.destroyAllWindows()
face_mesh.close()

c:\Users\super\Desktop\MACHINE LEARNING\MACHINE LEARNING 2026\Vision_computadora\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


---

# 💡 Ideas de proyectos con MediaPipe

| Proyecto | Módulo | Descripción |
|----------|--------|--------------|
| Reconocimiento de gestos | `hands` | Detectar gestos (pulgar arriba, ok, puño) para controlar apps |
| Contador de ejercicios | `pose` | Contar flexiones o sentadillas analizando ángulos articulares |
| Control de cursor | `hands` | Mover el mouse con la posición de los dedos |
| Filtros faciales en AR | `face_mesh` | Superponer efectos visuales sobre el rostro en tiempo real |
| Análisis de postura | `pose` | Evaluar la técnica en deporte, danza o fisioterapia |
| Juegos interactivos | `hands` / `pose` | Controlar juegos con movimientos corporales |
| Monitoreo de atención | `face_detection` | Detectar si la persona mira hacia la cámara |
| Lenguaje de señas | `hands` | Traducir señas a texto en tiempo real |
| Sistema de seguridad | `pose` | Alertar cuando se detecta presencia de personas |

---

## 📚 Recursos adicionales

- [Documentación oficial de MediaPipe](https://developers.google.com/mediapipe)
- [MediaPipe en GitHub](https://github.com/google/mediapipe)
- [Guía de landmarks de manos](https://developers.google.com/mediapipe/solutions/vision/hand_landmarker)
- [Guía de landmarks de pose](https://developers.google.com/mediapipe/solutions/vision/pose_landmarker)